# 02 – Benchmark: TsHARP against ECOSTRESS on all clear nights

For every clear night (at least 70 % of AOI land observed cloud-free by ECOSTRESS):

- **Reference:** ECOSTRESS LST, water and clouds masked, averaged onto the 70 m grid.
- **Coarse input:** the MODIS night pass (Terra or Aqua) closest in time to ECOSTRESS, on the 980 m grid.
- **Methods:** (a) MODIS upsampled without sharpening, (b) TsHARP (Agam et al. 2007) with that summer's Sentinel-2 NDVI.

Metrics per night: RMSE, bias-removed RMSE (`crmse`, the spatial pattern error), MAE, bias, R² and r.

Outputs: `data/benchmark_tsharp.csv`, `figures/02_benchmark_summary.png`, `figures/02_tsharp_example.png`.

Run `01_data_check.ipynb` first in the same session, or this notebook recomputes the night list.

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/KeremBasimc/istanbul-night-heat-island"
if "google.colab" in sys.modules or Path("/content").exists():
    ROOT = Path("/content/istanbul-night-heat-island")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data"
FIGS = ROOT / "figures"
DATA.mkdir(exist_ok=True); FIGS.mkdir(exist_ok=True)

import earthaccess
# Tries, in order: environment variables, ~/.netrc, then an interactive prompt.
print("Earthdata login ok:", earthaccess.login().authenticated)

## Clear nights

In [ ]:
import numpy as np
import pandas as pd
from nhi import data, downscale as ds

nights_csv = DATA / "ecostress_nights.csv"
if nights_csv.exists():
    per_night = pd.read_csv(nights_csv, parse_dates=["local"])
else:
    g = data.search_ecostress("2018-07-01", "2025-09-30")
    g = data.add_clear_fraction(g[g.is_night], DATA / "ecostress")
    per_night = data.best_granule_per_night(g)
    per_night.to_csv(nights_csv, index=False)

nights = per_night[per_night.clear_frac >= 0.7].reset_index(drop=True)
print(len(nights), "clear nights")
coarse, fine = data.grids()

# One land mask for every method: ESA WorldCover water < 50 % (ECOSTRESS misses some lakes).
from nhi import features as fe
land = fe.worldcover_fractions(fine)["water"] < 0.5

## Sentinel-2 summer NDVI per year (cached as GeoTIFF)
Water (NDVI ≤ 0 or WorldCover water) is masked, so TsHARP is fitted on land only.

In [ ]:
import rioxarray  # noqa: F401

_ndvi = {}
def ndvi_land(year):
    if year not in _ndvi:
        path = DATA / f"ndvi_{year}.tif"
        if path.exists():
            arr = rioxarray.open_rasterio(path).squeeze("band", drop=True).values
        else:
            nd = data.sentinel2_ndvi(year, fine)
            nd.rio.write_crs(fine.crs.to_wkt(), inplace=True).rio.to_raster(path)
            arr = nd.values
        _ndvi[year] = np.where((arr > 0) & land, arr, np.nan)
    return _ndvi[year]

## Run both methods on every clear night

In [ ]:
import time

rows, kept = [], {}
t0 = time.time()
for r in nights.itertuples():
    paths = data.download([r.url_LST, r.url_cloud, r.url_water], DATA / "ecostress")
    eco = np.where(land, data.ecostress_on_grid(*paths, fine).values, np.nan)
    passes = data.load_modis_night_passes(pd.Timestamp(r.night_date).date(), coarse, hdf_folder=DATA / "modis_hdf")
    pick = data.closest_modis_pass(passes, pd.Timestamp(r.local))
    if pick is None:
        continue
    platform, mp, gap = pick
    lst_c = mp["lst"].values
    nd = ndvi_land(pd.Timestamp(r.night_date).year)
    preds = {"MODIS upsampled": np.where(land, ds.upsample(lst_c), np.nan), "TsHARP": ds.tsharp(lst_c, nd)[0]}
    for name, pred in preds.items():
        m = ds.metrics(pred, eco)
        m.update(night_date=r.night_date, method=name, platform=platform, gap_h=round(gap, 2),
                 eco_local=r.local, clear_frac=round(r.clear_frac, 3))
        rows.append(m)
    kept[r.night_date] = {"eco": eco, "modis": lst_c, **preds}
print(f"{len(kept)} nights processed in {time.time() - t0:.0f} s")

res = pd.DataFrame(rows)
res.to_csv(DATA / "benchmark_tsharp.csv", index=False)
cols = ["rmse", "crmse", "mae", "bias", "r2", "r"]
res.groupby("method")[cols].agg(["mean", "median"]).round(2)

## Summary figure

In [ ]:
import matplotlib.pyplot as plt

methods = ["MODIS upsampled", "TsHARP"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, col, label in zip(axes, ["rmse", "crmse", "r"],
                          ["RMSE (°C)", "Bias-removed RMSE (°C)", "Correlation r"]):
    ax.boxplot([res.loc[res.method == m, col].dropna() for m in methods])
    ax.set_xticks([1, 2], methods)
    ax.set_title(label); ax.grid(axis="y", alpha=.3)
fig.suptitle(f"Against ECOSTRESS at 70 m, {res.night_date.nunique()} clear nights 2018–2025")
fig.savefig(FIGS / "02_benchmark_summary.png", dpi=150)
plt.show()

print("Time gap MODIS–ECOSTRESS (h): median", res.gap_h.median(), "| max", res.gap_h.max())
print(res.drop_duplicates("night_date").platform.value_counts().to_dict())

## Example night: ECOSTRESS vs MODIS upsampled vs TsHARP

In [ ]:
tsh = res[res.method == "TsHARP"].copy()
tsh["month"] = pd.to_datetime(tsh.night_date).dt.month
ex = tsh[tsh.month.isin([6, 7, 8, 9])].sort_values("clear_frac", ascending=False).iloc[0]
k = kept[ex.night_date]
vmin, vmax = np.nanpercentile(k["eco"], [2, 98])
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4), constrained_layout=True)
for ax, key, title in [(axes[0], "eco", "ECOSTRESS 70 m (reference)"),
                       (axes[1], "MODIS upsampled", f"MODIS {ex.platform.title()} upsampled"),
                       (axes[2], "TsHARP", "TsHARP 70 m")]:
    im = ax.imshow(k[key], cmap="inferno", vmin=vmin, vmax=vmax)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, label="Land surface temperature (°C)", shrink=0.85)
fig.suptitle(f"Night of {ex.night_date}: TsHARP RMSE {ex.rmse:.2f} °C, bias-removed {ex.crmse:.2f} °C, r {ex.r:.2f}")
fig.savefig(FIGS / "02_tsharp_example.png", dpi=150)
plt.show()

## Reading the numbers

- **Bias** mostly reflects the time gap between the MODIS pass and ECOSTRESS: the city keeps cooling during the night.
- **Bias-removed RMSE** and **r** show how well each method reproduces the spatial pattern at 70 m.
- The proposed method (notebook 03) must beat TsHARP on these metrics with night-specific predictors.